In [ ]:
import glob, sys, traceback
from pathlib import Path
import numpy as np, pandas as pd
CK = Path(glob.glob('/kaggle/input/**/f0_best.pt', recursive=True)[0]).parent
DATA = Path(glob.glob('/kaggle/input/**/train_series.csv', recursive=True)[0]).parent
sys.path.insert(0, str(CK)); sys.path.insert(0, str(CK / 'vjepa2'))
import infer as V
ser = pd.read_csv(DATA / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
print(ser.dtypes.to_dict())
for tag, uids in (('BAD', ["1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149", "1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604", "1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581", "1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903"]), ('GOOD', ["1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269", "1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052"])):
    for u in uids:
        rows = ser[ser.StudyInstanceUID == u]
        for vi, (plane, fluid) in enumerate(V.VIEWS5):
            cand = rows[(rows['Anatomical_Plane'] == plane) & (rows['Fluid_Sensitive'] == fluid)]
            msg = 'no series'
            for row in cand.itertuples(index=False):
                try:
                    st = V._load_source_slices(DATA / 'train_series' / u / str(row.SeriesInstanceUID))
                    msg = 'ok %s mean %.1f' % (None if st is None else st.shape, -1 if st is None else st.mean()); break
                except Exception:
                    msg = 'EXC ' + traceback.format_exc().strip().splitlines()[-1][:250]
            print(tag, u[-8:], plane, fluid, msg, flush=True)
